# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

## 1. Rebuild the Same Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/Users/ayush/Documents/rag-projects/Advanced_Rag_Codes/04_vector_stores')

In [3]:
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: /Users/ayush/Documents/rag-projects/Advanced_Rag_Codes/04_vector_stores/.env


In [5]:
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: /Users/ayush/Documents/rag-projects/Advanced_Rag_Codes/04_vector_stores/notebooks/chroma_langchain_db


In [6]:
# Use the same embedding model that was used to build the store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


## 2. Add Small Display Helpers

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [8]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [9]:
stored_records["embeddings"].shape

(8, 1536)

In [10]:
print_stored_documents(stored_records)

Total documents in collection: 8

1. id=8174f844-07e4-462b-84ee-a61d85a0f9a5
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human rea...

2. id=8626d7ca-a258-44c8-8b2e-fbbd473df263
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.

3. id=c86becab-5005-4cb7-b170-dabe5e1a1cb1
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.

4. id=3da1442d-7531-49fd-91ef-57ba511b0953
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language m...

5. id=850b6851-a9af-41e8-a8c5-a5e5735aa5b6
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model ge...

6. id=03e53a54-5543-45e1-813e-f9f0897ebae8
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic 